In [1]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

DATA = PROJECT_ROOT / "data"
OUTPUT_FIGURES = PROJECT_ROOT / "output" / "figures"
OUTPUT_TABLES = PROJECT_ROOT / "output" / "tables"

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm

# --------------------------------------------------
# 1. Load data
# --------------------------------------------------

top_games = pd.read_csv(DATA/ "top_steam_games.csv")
steam_history = pd.read_csv(DATA / "steamcharts_top_games_history.csv")
all_data = pd.read_csv(DATA / "all_data.csv")

# --------------------------------------------------
# 2. Clean column names
# --------------------------------------------------

top_games.columns = top_games.columns.str.lower().str.strip()
steam_history.columns = steam_history.columns.str.lower().str.strip()
all_data.columns = all_data.columns.str.lower().str.strip()


# --------------------------------------------------
# 3. Create latest SteamCharts observation per game
# --------------------------------------------------

steam_history["month"] = pd.to_datetime(steam_history["month"], errors="coerce")

latest = (steam_history.sort_values("month").groupby("app_id").tail(1).copy()
)

latest = latest.rename(columns={"avg_players": "current_players"})

latest["retention"] = latest["current_players"] / latest["peak_players"]

# --------------------------------------------------
# 4. Merge with metadata
# --------------------------------------------------

all_data = all_data.rename(columns={"appid": "app_id"})

df = latest.merge(all_data, on="app_id", how="left", suffixes=("", "_metadata")
)


# --------------------------------------------------
# 5. Manual live-service classification
# --------------------------------------------------

live_service_app_ids = [
    730,      # Counter-Strike 2
    570,      # Dota 2
    578080,   # PUBG
    1172470,  # Apex Legends
    1085660,  # Destiny 2
    2676230,  # FiveM
    440,      # Team Fortress 2
    252490,   # Rust
    359550,   # Rainbow Six Siege
    230410,   # Warframe
    2507950   # Delta Force
]

df["live_service"] = df["app_id"].isin(live_service_app_ids).astype(int)


# --------------------------------------------------
# 6. Extra variables
# --------------------------------------------------

df["log_current_players"] = np.log(df["current_players"])


# --------------------------------------------------
# OUTPUT 1: Summary statistics table
# --------------------------------------------------

summary_table = (
    df.groupby("live_service")
    .agg(
        games=("app_id", "count"),
        mean_average_players=("current_players", "mean"),
        median_average_players=("current_players", "median"),
        mean_peak_players=("peak_players", "mean"),
        mean_retention=("retention", "mean"),
    )
    .reset_index()
)

summary_table["live_service"] = summary_table["live_service"].map({ 0: "Traditional", 1: "Live service"})

summary_table.to_csv(OUTPUT_TABLES / "summary_by_model.csv", index=False)
print(summary_table)


# --------------------------------------------------
# OUTPUT 2: Bar chart of current players
# --------------------------------------------------

plot_df = df.sort_values("current_players", ascending=False).head(30)

plt.figure(figsize=(10, 8))
plt.barh(plot_df["game_name"], plot_df["current_players"])
plt.gca().invert_yaxis()
plt.xlabel("Peak players")
plt.ylabel("Game")
plt.title("Most-played Steam games in the sample")
ax = plt.gca()
plt.tight_layout()
plt.savefig(OUTPUT_FIGURES / "top_games_current_players.png", dpi=300)
plt.close()
# --------------------------------------------------
# OUTPUT 3: Share of top games that are live service
# --------------------------------------------------

share_df = df["live_service"].value_counts().sort_index()
share_df.index = ["Traditional", "Live service"]

plt.figure(figsize=(6, 4))
plt.bar(share_df.index, share_df.values)
plt.ylabel("Number of games")
plt.title("How many top Steam games are live-service?")
plt.tight_layout()
plt.savefig(OUTPUT_FIGURES / "live_service_share.png", dpi=300)
plt.close()


# --------------------------------------------------
# OUTPUT 4: Retention comparison boxplot
# --------------------------------------------------

traditional_retention = df.loc[df["live_service"] == 0, "retention"].dropna()
live_retention = df.loc[df["live_service"] == 1, "retention"].dropna()

plt.figure(figsize=(6, 4))
plt.boxplot([traditional_retention, live_retention], labels=["Traditional", "Live service"])
plt.ylabel("Retention: Current Players / Peak Players")
plt.title("Player retention by game model")
plt.tight_layout()
plt.savefig(OUTPUT_FIGURES / "retention_boxplot.png", dpi=300)
plt.close()

# --------------------------------------------------
# OUTPUT 5: Regression table
# --------------------------------------------------

reg_df = df[["log_current_players", "retention", "live_service"]].dropna()

X = sm.add_constant(reg_df[["live_service"]])

model_players = sm.OLS(reg_df["log_current_players"], X).fit()
model_retention = sm.OLS(reg_df["retention"], X).fit()

with open(OUTPUT_TABLES / "regression_results.txt", "w") as f:
    f.write("MODEL 1: log_current_players on live_service\n")
    f.write(model_players.summary().as_text())
    f.write("\n\n")
    f.write("MODEL 2: retention on live_service\n")
    f.write(model_retention.summary().as_text())

print(model_players.summary())
print(model_retention.summary())

   live_service  games  mean_average_players  median_average_players  \
0   Traditional     12          75715.110833               62606.535   
1  Live service      8         281740.413750              127923.615   

   mean_peak_players  mean_retention  
0         151148.750        0.507361  
1         524219.125        0.541288  
                             OLS Regression Results                            
Dep. Variable:     log_current_players   R-squared:                       0.348
Model:                             OLS   Adj. R-squared:                  0.311
Method:                  Least Squares   F-statistic:                     9.594
Date:                 Fri, 08 May 2026   Prob (F-statistic):            0.00622
Time:                         10:22:08   Log-Likelihood:                -20.401
No. Observations:                   20   AIC:                             44.80
Df Residuals:                       18   BIC:                             46.79
Df Model:                 

/tmp/ipykernel_113402/220425928.py:145: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  plt.boxplot([traditional_retention, live_retention], labels=["Traditional", "Live service"])
